### Frame Extraction

In [3]:
import av
import os
import numpy as np
from PIL import Image
from tqdm import tqdm

CHOLEC80_DIR = "Cholec80/cholec80"
VIDEO_DIR = os.path.join(CHOLEC80_DIR, "videos")
PHASE_DIR = os.path.join(CHOLEC80_DIR, "phase_annotations")
FRAME_DIR = "cholec80_frames"   # extracted frames go here
EXTRACT_FPS = 1         # 1 frame per second
SAMPLE_EVERY = 25       # 25fps/1fps = every 25th frame

PHASE_MAP = {
    'Preparation':             0,
    'CalotTriangleDissection': 1,
    'ClippingCutting':         2,
    'GallbladderDissection':   3,
    'GallbladderPackaging':    4,
    'CleaningCoagulation':     5,
    'GallbladderRetraction':   6,
}

In [4]:
# frame number -> phase lebel (int)
def parse_phase_annotation(video_id):
    path = os.path.join(PHASE_DIR, f"video{video_id:02d}-phase.txt")
    labels = {}

    with open(path) as f:
        next(f)     # skip header
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) == 2:
                frame_num = int(parts[0])
                phase_str = parts[1].strip()
                if phase_str in PHASE_MAP:
                    labels[frame_num] = PHASE_MAP[phase_str]
            
    return labels

In [6]:
# extract frames
extraction_log = {}     #   video_id -> list of (frame_path, phase_label)

for video_id in tqdm(range(1, 81), desc="Videos"):
    video_path = os.path.join(VIDEO_DIR, f"video{video_id:02d}.mp4")
    if not os.path.exists(video_path):
        print(f"Missing: video{video_id:02d}.mp4 - skipping")
        continue

    # load phase labels for this video
    phase_labels = parse_phase_annotation(video_id)

    # output folder per video
    vid_frame_dir = os.path.join(FRAME_DIR, f"video{video_id:02d}")
    os.makedirs(vid_frame_dir, exist_ok=True)

    video_log = []
    container = av.open(video_path)

    for frame_idx, frame in enumerate(container.decode(video=0)):
        # extract 1fps from 25fps (every 25th frame)
        if frame_idx % SAMPLE_EVERY != 0:
            continue

        # get phase label for this phrame
        phase = phase_labels.get(frame_idx, -1)
        if phase == -1:
            continue # no annotations for this frame

        # save as JPEG

        img = frame.to_ndarray(format='rgb24')
        img_pil = Image.fromarray(img)
        img_small = img_pil.resize((224,224), Image.LANCZOS)

        frame_name = f"f{frame_idx:06d}.jpg"
        frame_path = os.path.join(vid_frame_dir, frame_name)
        img_small.save(frame_path, quality=90)

        video_log.append((frame_path, phase))

    container.close()
    extraction_log[video_id] = video_log

total_frames = sum(len(v) for v in extraction_log.values())
print(f"\n{'='*50}")
print(f"Extraction complete")
print(f"Videos processed:  {len(extraction_log)}")
print(f"Total frames:      {total_frames:,}")
print(f"\nPhase distribution:")

phase_counts = {v: 0 for v in PHASE_MAP.values()}
for video_log in extraction_log.values():
    for _, phase in video_log:
        phase_counts[phase] += 1

phase_names = {v: k for k, v in PHASE_MAP.items()}
for phase_id, count in sorted(phase_counts.items()):
    pct = count / total_frames * 100
    print(f"  {phase_names[phase_id]:<30} {count:>6,} frames ({pct:.1f}%)")

# save the log for later use
import json
log_serializable = {
    str(k): v for k, v in extraction_log.items()
}
with open("cholec80_frame_log.json", "w") as f:
    json.dump(log_serializable, f)
print(f"\nFrame log saved to: cholec80_frame_log.json")
print(f"Disk usage: ~{total_frames * 50 / 1024 / 1024:.1f} MB estimated")

Videos: 100%|██████████| 80/80 [1:44:57<00:00, 78.72s/it] 


Extraction complete
Videos processed:  80
Total frames:      184,578

Phase distribution:
  Preparation                     8,574 frames (4.6%)
  CalotTriangleDissection        74,826 frames (40.5%)
  ClippingCutting                14,080 frames (7.6%)
  GallbladderDissection          58,433 frames (31.7%)
  GallbladderPackaging            7,618 frames (4.1%)
  CleaningCoagulation            14,335 frames (7.8%)
  GallbladderRetraction           6,712 frames (3.6%)

Frame log saved to: cholec80_frame_log.json
Disk usage: ~8.8 MB estimated
